In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "backend/src/plexa/mobility/modeling").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the Plexa clone.")
DATA_ROOT = PROJECT_ROOT / "data" / "mobility"


In [1]:
import pandas as pd, numpy as np
OUT = str(DATA_ROOT / "clean")
A = pd.read_parquet(f"{OUT}/auto_hourly_v4.parquet")
H = pd.read_parquet(f"{OUT}/hist_hourly_v2.parquet")

# 1. Schema, nulls, ranges
for name, X in [("automated v4", A), ("historical v2", H)]:
    print(f"\n=== {name} ===")
    print("rows:", len(X), "| segments:", X["segment_id"].nunique(),
          "| range:", X["hour_start"].min(), "->", X["hour_start"].max())
    print(pd.DataFrame({"dtype": X.dtypes.astype(str), "nulls": X.isna().sum(),
                        "null_%": (X.isna().mean() * 100).round(1)}).to_string())

print("\nOnly in automated:", sorted(set(A.columns) - set(H.columns)))
print("Only in historical:", sorted(set(H.columns) - set(A.columns)))

# 2. Key duplicates, flags, LION keys
kk = ["segment_id", "direction", "hour_start"]
print("\nDuplicate (segment, direction, hour): automated", int(A.duplicated(kk).sum()),
      "| historical", int(H.duplicated(kk).sum()))
for name, X in [("A", A), ("H", H)]:
    fl = [c for c in ["partial_day", "covid", "sparse_year", "dst_issue", "identity_conflict",
                      "zero_to_missing"] if c in X.columns]
    print(name, "flag shares:", X[fl].mean().round(4).to_dict())
    print(name, "lion_key_used:", X["lion_key_used"].value_counts().to_dict())

# 3. Overlap preview (nothing is changed)
m = A[kk + ["volume"]].merge(H[kk + ["volume", "road_type"]], on=kk, suffixes=("_auto", "_hist"))
print("\nOverlapping segment-direction-hours:", len(m), "| segments:", m["segment_id"].nunique())
print("Historical road_type in the overlap:", m["road_type"].value_counts().to_dict())
m = m[m["volume_hist"] > 0]
r = m["volume_auto"] / m["volume_hist"]
print(r.describe(percentiles=[.05, .25, .5, .75, .95]))
print("Within +-10%:", round(((r - 1).abs() <= 0.10).mean(), 4))


=== automated v4 ===
rows: 444140 | segments: 2843 | range: 2000-01-01 01:00:00 -> 2026-02-03 23:00:00
                                                dtype  nulls  null_%
request_id                                      int64      0     0.0
segment_id                                      int64      0     0.0
direction                                      object      0     0.0
hour_start                             datetime64[ns]      0     0.0
orig_interval_min                               int64      0     0.0
vol_sum                                       float64      0     0.0
n_valid                                         int64      0     0.0
volume                                        float64      0     0.0
date                                   datetime64[ns]      0     0.0
hours_in_day                                    int64      0     0.0
partial_day                                      bool      0     0.0
covid                                            bool      0     0.0

In [1]:
import pandas as pd, numpy as np

OUT = str(DATA_ROOT / "clean")
A = pd.read_parquet(f"{OUT}/auto_hourly_v4.parquet")
H = pd.read_parquet(f"{OUT}/hist_hourly_v2.parquet")
kk = ["segment_id", "direction", "hour_start"]

# ---- Check 1: what are the historical duplicate keys? ----
d = H[H.duplicated(kk, keep=False)].copy()
g = d.groupby(kk)["volume"]
print("Rows in duplicate groups:", len(d), "| groups:", g.ngroups)
print("Group sizes:", g.size().value_counts().to_dict())
print("Road-type combinations:",
      d.groupby(kk)["road_type"].agg(lambda s: "+".join(sorted(s))).value_counts().to_dict())
print("Groups where all volumes are identical (likely copies):", int((g.nunique() == 1).sum()))
print("Groups where volumes differ:", int((g.nunique() > 1).sum()))
ratio = (g.max() / g.min().replace(0, np.nan)).dropna()
print("Max/min volume ratio within a group:")
print(ratio.describe(percentiles=[.05, .25, .5, .75, .95]).round(3))
print("Groups by year:", d.groupby(d["hour_start"].dt.year)[kk[0]].nunique().to_dict())
print("\nExample groups:")
print(d.sort_values(kk).head(12)[kk + ["road_type", "volume", "lion_key_used"]].to_string())

# ---- Check 2: overlap with automated, by road type ----
m_all = A[kk + ["volume"]].merge(H[kk + ["volume", "road_type"]], on=kk,
                                 suffixes=("_auto", "_hist"))
print("\nOverlap rows (all historical road types):", len(m_all),
      "| segments:", m_all["segment_id"].nunique())
print(m_all.groupby("road_type").size())

m_all = m_all[m_all["volume_hist"] > 0]
r = m_all["volume_auto"] / m_all["volume_hist"]
print("Ratio auto/hist by road type:")
print(m_all.assign(r=r).groupby("road_type")["r"]
           .agg(["median", "size", lambda s: round(((s - 1).abs() <= 0.10).mean(), 4)])
           .rename(columns={"<lambda_0>": "within_10pct"}))

# ---- Check 3: for overlapping duplicate groups, which historical value matches automated? ----
dup_keys = set(map(tuple, d[kk].drop_duplicates().values.tolist()))
ov = m_all[[tuple(x) in dup_keys for x in m_all[kk].values.tolist()]].copy()
if len(ov):
    gg = d.groupby(kk)["volume"]
    stats = pd.DataFrame({"s_sum": gg.sum(), "s_max": gg.max(), "s_min": gg.min()}).reset_index()
    ov = ov.merge(stats, on=kk)
    ov["to_sum"] = (ov["volume_auto"] / ov["s_sum"]).round(2)
    ov["to_max"] = (ov["volume_auto"] / ov["s_max"]).round(2)
    ov["to_min"] = (ov["volume_auto"] / ov["s_min"]).round(2)
    print("\nOverlapping duplicate groups:", len(ov))
    print(ov[["to_sum", "to_max", "to_min"]].describe(percentiles=[.25, .5, .75]).round(3))
    print("Closest reference (share of rows where automated is nearest to):",
          pd.DataFrame({"sum": (ov["to_sum"] - 1).abs(), "max": (ov["to_max"] - 1).abs(),
                        "min": (ov["to_min"] - 1).abs()}).idxmin(axis=1).value_counts().to_dict())
else:
    print("\nNo overlapping historical duplicate groups.")

Rows in duplicate groups: 5856 | groups: 2928
Group sizes: {2: 2928}
Road-type combinations: {'main+service': 2064, 'service+unspecified': 864}
Groups where all volumes are identical (likely copies): 0
Groups where volumes differ: 2928
Max/min volume ratio within a group:
count    2928.000
mean       11.656
std        11.977
min         1.232
5%          2.110
25%         3.937
50%         7.332
75%        14.959
95%        35.590
max       215.167
Name: volume, dtype: float64
Groups by year: {2012: 7, 2013: 7}

Example groups:
        segment_id direction          hour_start    road_type  volume lion_key_used
391602       56804        NB 2013-02-02 00:00:00      service   100.0     SegmentID
391746       56804        NB 2013-02-02 00:00:00  unspecified  2688.0     SegmentID
391603       56804        NB 2013-02-02 01:00:00      service    75.0     SegmentID
391747       56804        NB 2013-02-02 01:00:00  unspecified  1751.0     SegmentID
391604       56804        NB 2013-02-02 02:00:

In [2]:
OUT = str(DATA_ROOT / "clean")
H = pd.read_parquet(f"{OUT}/hist_hourly_v2.parquet")

# 1. Which segment-direction combinations carry which series?
H2 = H[["segment_id", "direction", "road_type", "hour_start", "volume", "date"]].copy()
sets = H2.groupby(["segment_id", "direction"])["road_type"].agg(lambda s: "+".join(sorted(s.unique())))
print("Series combinations per segment-direction:")
print(sets.value_counts().to_string())

# 2. Median level per series, per segment-direction
med = (H2.groupby(["segment_id", "direction", "road_type"])["volume"].median()
         .unstack("road_type"))
print("\nSegment-directions carrying each series:")
print(med.notna().sum().to_string())

# 3. Pairwise level comparisons
for a, b in [("main", "unspecified"), ("service", "unspecified"), ("main", "service")]:
    x = med[[a, b]].dropna()
    if len(x) == 0:
        print(f"\n{a} vs {b}: no segment-directions carry both")
        continue
    ratio = x[a] / x[b].replace(0, np.nan)
    print(f"\n{a} vs {b}: segment-directions={len(x)} | median ratio={ratio.median():.3f} "
          f"| share where {a} > {b}={(x[a] > x[b]).mean():.2f}")
    print(ratio.describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())

# 4. Which segments do the duplicate series come from, and when?
dup = H2[H2.duplicated(["segment_id", "direction", "hour_start"], keep=False)]
print("\nSegments with duplicate series:", sorted(dup["segment_id"].unique().tolist()))
print(dup.groupby("segment_id")["date"].agg(["min", "max", "nunique"]).to_string())

Series combinations per segment-direction:
road_type
unspecified                 2999
main+service                   7
service+unspecified            5
main+service+unspecified       2

Segment-directions carrying each series:
road_type
main              9
service          14
unspecified    3006

main vs unspecified: segment-directions=2 | median ratio=3.227 | share where main > unspecified=1.00
count    2.000
mean     3.227
std      2.359
min      1.559
10%      1.893
25%      2.393
50%      3.227
75%      4.061
90%      4.561
max      4.894

service vs unspecified: segment-directions=7 | median ratio=0.265 | share where service > unspecified=0.14
count    7.000
mean     0.395
std      0.419
min      0.063
10%      0.159
25%      0.242
50%      0.265
75%      0.313
90%      0.744
max      1.325

main vs service: segment-directions=9 | median ratio=10.726 | share where main > service=1.00
count     9.000
mean     13.733
std       9.281
min       3.695
10%       5.453
25%       6.233
50

In [3]:
OUT = str(DATA_ROOT / "clean")
H = pd.read_parquet(f"{OUT}/hist_hourly_v2.parquet", columns=["segment_id", "road_type", "hour_start"])
print(H["road_type"].value_counts().to_string())
print("Share of rows that are main or service:",
      round(H["road_type"].isin(["main", "service"]).mean() * 100, 2), "%")

road_type
unspecified    1002428
service           3024
main              2064
Share of rows that are main or service: 0.51 %


In [4]:
OUT = str(DATA_ROOT / "clean")
A = pd.read_parquet(f"{OUT}/auto_hourly_v4.parquet")
H = pd.read_parquet(f"{OUT}/hist_hourly_v2.parquet")

LION_COLS = ["POSTED_SPEED", "Number_Travel_Lanes", "Number_Park_Lanes", "Number_Total_Lanes",
             "StreetWidth_Min", "StreetWidth_Max", "SHAPE_Length", "TrafDir", "TRUCK_ROUTE_TYPE",
             "Snow_Priority", "BikeLane", "LBoro", "RBoro", "lon", "lat", "lion_wkt_2263"]
BORO = {1: "Manhattan", 2: "Bronx", 3: "Brooklyn", 4: "Queens", 5: "Staten Island"}

# 1. Per-segment LION attributes, one row per segment
a_att = A.drop_duplicates("segment_id").set_index("segment_id")[LION_COLS + ["lion_key_used"]]
h_ok = H[~H["identity_conflict"]]
h_att = h_ok.drop_duplicates("segment_id").set_index("segment_id")[LION_COLS + ["lion_key_used"]]
print("automated segments:", len(a_att), "| historical segments with a valid LION row:", len(h_att))

# Internal consistency: LION attributes should be constant within a segment
for name, X in [("automated", A), ("historical", h_ok)]:
    v = X.groupby("segment_id")[LION_COLS].nunique(dropna=True).gt(1).sum()
    print(f"{name}: columns varying within a segment (should be empty):", v[v > 0].to_dict())

# 2. Overlap consistency: do the two sources describe the same LION road?
common = a_att.index.intersection(h_att.index)
print("\nSegments in both sources (with valid historical LION):", len(common))
both_ok = {}
for c in LION_COLS + ["lion_key_used"]:
    x, y = a_att.loc[common, c], h_att.loc[common, c]
    both_ok[c] = int(((x != y) & x.notna() & y.notna()).sum())
print("Disagreements on shared segments:", {k: v for k, v in both_ok.items() if v} or "none")

# 3. Build the segment table over the union of segments
all_ids = sorted(set(A["segment_id"]) | set(H["segment_id"]))
seg = pd.DataFrame(index=pd.Index(all_ids, name="segment_id"))
seg["in_automated"] = seg.index.isin(A["segment_id"])
seg["in_historical"] = seg.index.isin(H["segment_id"])
print("\nUnion of segments:", len(seg), "| expected 4673")
print(seg.groupby(["in_automated", "in_historical"]).size().to_string())
assert seg.index.is_unique

# Attributes: automated first, historical fallback
att_src = pd.concat([a_att.assign(attr_source="automated"),
                     h_att.loc[~h_att.index.isin(a_att.index)].assign(attr_source="historical")])
seg = seg.join(att_src)

# Identity conflicts (historical rows only)
conf = H.groupby("segment_id")["identity_conflict"].agg(hist_rows="size", conflict_rows="sum")
seg = seg.join(conf)
seg["conflict_share"] = (seg["conflict_rows"] / seg["hist_rows"]).round(3)
seg["identity_conflict_all"] = seg["conflict_share"] == 1
seg.loc[seg["identity_conflict_all"] & seg["lion_key_used"].isna(), "lion_key_used"] = "identity_conflict"
seg["lion_key_used"] = seg["lion_key_used"].fillna("none")
print("\nSegments with any identity-conflict rows:", int((seg["conflict_rows"].fillna(0) > 0).sum()))
print("Segments where every historical row conflicts:", int(seg["identity_conflict_all"].sum()))
print("LION key by segment:\n", seg["lion_key_used"].value_counts().to_string())

# Borough: LION code first, automated Boro as fallback
seg["lion_boro"] = seg["LBoro"].map(lambda x: BORO.get(int(x)) if pd.notna(x) else np.nan)
seg["automated_boro"] = A.drop_duplicates("segment_id").set_index("segment_id")["Boro"]
both_b = seg.dropna(subset=["lion_boro", "automated_boro"])
print("\nBorough: LION vs automated disagreements:", int((both_b["lion_boro"] != both_b["automated_boro"]).sum()),
      "of", len(both_b))
seg["borough"] = seg["lion_boro"].fillna(seg["automated_boro"])
print(seg["borough"].value_counts(dropna=False).to_string())

# 4. Time coverage per source
spans = pd.concat([
    A.groupby("segment_id")["hour_start"].agg(a_first="min", a_last="max", a_hours="size"),
    H.groupby("segment_id")["hour_start"].agg(h_first="min", h_last="max", h_hours="size")], axis=1)
seg = seg.join(spans)
seg["first_hour"] = seg[["a_first", "h_first"]].min(axis=1)
seg["last_hour"] = seg[["a_last", "h_last"]].max(axis=1)

# 5. Series present in historical
ser = H.groupby("segment_id")["road_type"].agg(lambda s: "+".join(sorted(s.unique())))
seg = seg.join(ser.rename("hist_series"))
seg["has_minor_series"] = seg["hist_series"].fillna("").str.contains("main|service")
print("\nSegments with main or service series:", int(seg["has_minor_series"].sum()), "(expected 8)")

seg = seg.reset_index()
print("\nFinal segment table:", seg.shape)
seg.to_parquet(f"{OUT}/segment_table_v1.parquet", index=False)

automated segments: 2843 | historical segments with a valid LION row: 1926
automated: columns varying within a segment (should be empty): {}
historical: columns varying within a segment (should be empty): {}

Segments in both sources (with valid historical LION): 307
Disagreements on shared segments: {'SHAPE_Length': 47, 'lion_key_used': 306}

Union of segments: 4488 | expected 4673
in_automated  in_historical
False         True             1645
True          False            2534
              True              309

Segments with any identity-conflict rows: 48
Segments where every historical row conflicts: 28
LION key by segment:
 lion_key_used
SegmentID              3801
LegacyID                446
spatial                  94
none                     84
automated_validated      37
identity_conflict        26

Borough: LION vs automated disagreements: 16 of 2794
borough
Brooklyn         1297
Queens           1222
Bronx             797
Manhattan         788
Staten Island     317
NaN   

In [5]:
OUT = str(DATA_ROOT / "clean")
A = pd.read_parquet(f"{OUT}/auto_hourly_v4.parquet")
H = pd.read_parquet(f"{OUT}/hist_hourly_v2.parquet")
seg = pd.read_parquet(f"{OUT}/segment_table_v1.parquet")
cols = ["SHAPE_Length", "lion_wkt_2263", "lion_key_used"]

a_sl = A.drop_duplicates("segment_id").set_index("segment_id")[cols]
h_sl = H[~H["identity_conflict"]].drop_duplicates("segment_id").set_index("segment_id")[cols]
common = a_sl.index.intersection(h_sl.index)

# 1. SHAPE_Length differences
diff = a_sl.loc[common, "SHAPE_Length"] - h_sl.loc[common, "SHAPE_Length"]
print("SHAPE_Length difference stats:"); print(diff.describe().to_string())
bad = diff[diff.abs() > 1e-6].index
print(pd.DataFrame({"auto": a_sl.loc[bad, "SHAPE_Length"], "hist": h_sl.loc[bad, "SHAPE_Length"],
                    "diff": diff[bad]}).head(10).to_string())

# 2. Key label crosstab (is the one non-matching segment a real difference?)
print(pd.crosstab(a_sl.loc[common, "lion_key_used"], h_sl.loc[common, "lion_key_used"]).to_string())

# 3. Borough disagreements
bd = seg.dropna(subset=["lion_boro", "automated_boro"])
bd = bd[bd["lion_boro"] != bd["automated_boro"]]
print("\nBorough disagreements:", len(bd))
print(bd[["segment_id", "lion_boro", "automated_boro", "LBoro", "lion_key_used"]].to_string())

# 4. Fully conflicting segments that still have automated attributes
x = seg[seg["identity_conflict_all"] & (seg["attr_source"] == "automated")]
print("\nFully conflicting with automated attributes:", len(x))
print(x[["segment_id", "lion_key_used", "conflict_rows", "hist_rows"]].to_string())

SHAPE_Length difference stats:
count    3.060000e+02
mean     2.426527e-15
std      2.553991e-14
min     -2.273737e-13
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.136868e-13
Empty DataFrame
Columns: [auto, hist, diff]
Index: []
lion_key_used  automated_validated  none
lion_key_used                           
LegacyID                        19     0
SegmentID                      274     0
none                             0     1
spatial                         13     0

Borough disagreements: 16
      segment_id      lion_boro automated_boro  LBoro lion_key_used
59          5946  Staten Island       Brooklyn    5.0     SegmentID
230        17640       Brooklyn         Queens    3.0     SegmentID
1053       36764      Manhattan         Queens    1.0     SegmentID
1060       36905      Manhattan         Queens    1.0     SegmentID
1342       45270         Queens       Brooklyn    4.0     SegmentID
1350       45393         Queens       Brooklyn    4.0     

In [6]:
OUT = str(DATA_ROOT / "clean")
A = pd.read_parquet(f"{OUT}/auto_hourly_v4.parquet", columns=["segment_id", "Boro", "LBoro", "RBoro"])
seg = pd.read_parquet(f"{OUT}/segment_table_v1.parquet")

bd_ids = seg.loc[(seg["lion_boro"].notna()) & (seg["automated_boro"].notna())
                 & (seg["lion_boro"] != seg["automated_boro"]), "segment_id"]
x = (A.drop_duplicates("segment_id").set_index("segment_id")
       .loc[bd_ids.values, ["Boro", "LBoro", "RBoro"]])
BORO = {1.0: "Manhattan", 2.0: "Bronx", 3.0: "Brooklyn", 4.0: "Queens", 5.0: "Staten Island"}
x["RBoro_name"] = x["RBoro"].map(BORO)
x["matches_right_side"] = x["RBoro_name"] == x["Boro"]
print(x.to_string())
print("\nRight-side borough matches the automated borough:", int(x["matches_right_side"].sum()), "of", len(x))
print("Share of all automated segments where LBoro != RBoro:",
      round((A.drop_duplicates("segment_id")["LBoro"] != A.drop_duplicates("segment_id")["RBoro"]).mean() * 100, 2), "%")

                Boro  LBoro  RBoro     RBoro_name  matches_right_side
segment_id                                                           
5946        Brooklyn    5.0    5.0  Staten Island               False
17640         Queens    3.0    3.0       Brooklyn               False
36764         Queens    1.0    1.0      Manhattan               False
36905         Queens    1.0    1.0      Manhattan               False
45270       Brooklyn    4.0    4.0         Queens               False
45393       Brooklyn    4.0    4.0         Queens               False
45402       Brooklyn    4.0    4.0         Queens               False
51124       Brooklyn    4.0    4.0         Queens               False
66420         Queens    3.0    3.0       Brooklyn               False
114477        Queens    1.0    1.0      Manhattan               False
136594      Brooklyn    4.0    4.0         Queens               False
141027      Brooklyn    4.0    4.0         Queens               False
141367      Brooklyn

In [7]:
RAW = str(DATA_ROOT / "Automated_Traffic_Volume_Counts_20260928.csv")
raw = pd.read_csv(RAW, usecols=["SegmentID", "Boro", "WktGeom", "street", "fromSt", "toSt"])
s = raw.drop_duplicates("SegmentID").set_index("SegmentID")
xy = s["WktGeom"].str.extract(r"POINT \(([-\d\.]+) ([-\d\.]+)\)").astype(float)
s["x"], s["y"] = xy[0].values, xy[1].values

seg = pd.read_parquet(str(DATA_ROOT / "clean/segment_table_v1.parquet"))
lion_b = seg.set_index("segment_id")["lion_boro"]

bad = [5946, 17640, 36764, 36905, 45270, 45393, 45402, 51124,
       66420, 114477, 136594, 141027, 141367, 145550, 155079, 158339]
ref = s.loc[~s.index.isin(bad)]          # other count sites, not in dispute
R = ref[["x", "y"]].to_numpy(dtype=float)
FT_TO_KM = 0.0003048

rows = []
for sid in bad:
    p = s.loc[sid, ["x", "y"]].to_numpy(dtype=float)
    dist = np.hypot(R[:, 0] - p[0], R[:, 1] - p[1])
    nn = np.argsort(dist)[:15]
    labels = ref["Boro"].to_numpy()[nn]
    vals, cnt = np.unique(labels, return_counts=True)
    rows.append({"segment_id": sid,
                 "automated_boro": s.loc[sid, "Boro"],
                 "lion_boro": lion_b.get(sid),
                 "neighbour_majority": vals[cnt.argmax()],
                 "neighbour_share": round(cnt.max() / 15, 2),
                 "nearest_km": round(dist[nn[0]] * FT_TO_KM, 3),
                 "street": s.loc[sid, "street"],
                 "from": s.loc[sid, "fromSt"],
                 "to": s.loc[sid, "toSt"]})

res = pd.DataFrame(rows)
res["agrees_with"] = np.select(
    [res["neighbour_majority"] == res["automated_boro"], res["neighbour_majority"] == res["lion_boro"]],
    ["automated", "lion"], default="neither")
print(res.to_string())
print("\nAgreement with neighbours:", res["agrees_with"].value_counts().to_dict())

    segment_id automated_boro      lion_boro neighbour_majority  neighbour_share  nearest_km                                              street                        from                         to agrees_with
0         5946       Brooklyn  Staten Island      Staten Island             1.00       0.891  CONEY ISL AVE & CORTELYOU RD BET AVE C & CORTELYOU               BENNINGTON ST               HILLCREST ST        lion
1        17640         Queens       Brooklyn           Brooklyn             1.00       0.247                62nd DR WB APPROACH TO JUNCTION BLVD                BAY RIDGE AV                      68 ST        lion
2        36764         Queens      Manhattan          Manhattan             0.87       0.444                                             MAIN ST                        W RD  PARK PEDESTRIAN  CROSSING        lion
3        36905         Queens      Manhattan          Manhattan             0.73       0.126                                             MAIN ST        

In [8]:

OUT = str(DATA_ROOT / "clean")
A = pd.read_parquet(f"{OUT}/auto_hourly_v4.parquet")
H = pd.read_parquet(f"{OUT}/hist_hourly_v2.parquet")
seg = pd.read_parquet(f"{OUT}/segment_table_v1.parquet")
KK = ["segment_id", "direction", "hour_start"]
EXPECTED = len(A) + len(H) - 22018
print("inputs: automated", len(A), "| historical", len(H), "| expected combined", EXPECTED)

# 1. Segment table v2: borough
MANUAL = {36764: "Manhattan", 36905: "Manhattan", 114477: "Manhattan"}   # Roosevelt Island "Main St"
seg2 = seg.copy()
seg2["boro_disagreement"] = (seg2["lion_boro"].notna() & seg2["automated_boro"].notna()
                             & (seg2["lion_boro"] != seg2["automated_boro"]))
seg2["borough"] = seg2["automated_boro"].fillna(seg2["lion_boro"])
man = seg2["segment_id"].isin(MANUAL)
seg2.loc[man, "borough"] = seg2.loc[man, "segment_id"].map(MANUAL)
seg2["borough_source"] = np.select([man, seg2["automated_boro"].notna(), seg2["lion_boro"].notna()],
                                   ["manual_override", "automated", "lion"], default="none")
print("borough sources:", seg2["borough_source"].value_counts().to_dict())
print("boro_disagreement segments:", int(seg2["boro_disagreement"].sum()))
seg2.to_parquet(f"{OUT}/segment_table_v2.parquet", index=False)

# 2. Overlap: historical rows already covered by automated
a_keys = A[KK].drop_duplicates()
Hm = H.merge(a_keys.assign(_in_auto=True), on=KK, how="left")
assert len(Hm) == len(H), "merge changed row count"
H["overlap_auto"] = Hm["_in_auto"].fillna(False).astype(bool).to_numpy()
print("\nHistorical rows overlapping automated:", int(H["overlap_auto"].sum()),
      H.loc[H["overlap_auto"], "road_type"].value_counts().to_dict())
assert set(H.loc[H["overlap_auto"], "road_type"]) <= {"unspecified"}, "main/service overlap found: stop"
Hk = H.loc[~H["overlap_auto"]].copy()

# 3. Stack both sources on common columns
def stack_common(df, source):
    d = df.reset_index(drop=True)
    return pd.DataFrame({
        "segment_id": d["segment_id"].astype("int64"),
        "direction": d["direction"],
        "road_type": d["road_type"] if "road_type" in d.columns else "unspecified",
        "hour_start": d["hour_start"],
        "hour_start_tz": d["hour_start_tz"],
        "volume": d["volume"].astype(float),
        "source": source,
        "study_id": d["request_id"] if "request_id" in d.columns else d["count_id"],
        "orig_interval_min": d["orig_interval_min"].astype("int64"),
        "dst_issue": d["dst_issue"].astype(bool),
        "identity_conflict": d["identity_conflict"].astype(bool) if "identity_conflict" in d.columns else False,
    })

C = pd.concat([stack_common(A, "automated"), stack_common(Hk, "historical")], ignore_index=True)
assert len(C) == EXPECTED, f"row count {len(C)} != expected {EXPECTED}"
print("combined rows:", len(C), "| expected:", EXPECTED)

# 4. Flags recomputed on the combined data
C["date"] = C["hour_start"].dt.normalize()
C["hours_in_day"] = C.groupby(["segment_id", "direction", "road_type", "date"])["volume"].transform("size")
C["partial_day"] = C["hours_in_day"] < 24
C["covid"] = C["hour_start"] >= pd.Timestamp("2020-03-01")
C["sparse_year"] = C["hour_start"].dt.year.isin([2000, 2006, 2007, 2008, 2013, 2018, 2021, 2026])
C["baseline_ok"] = C["road_type"].eq("unspecified") & ~C["identity_conflict"]

# 5. Checks and reconciliation
dup = int(C.duplicated(["segment_id", "direction", "road_type", "hour_start"], keep=False).sum())
print("\nDuplicate keys at (segment, direction, road_type, hour):", dup, "(expected 0)")
print("Source counts:", C["source"].value_counts().to_dict())
print("Road types:", C["road_type"].value_counts().to_dict())
print("Identity-conflict rows:", int(C["identity_conflict"].sum()), f"({C['identity_conflict'].mean():.2%})")
print("Baseline-eligible rows:", int(C["baseline_ok"].sum()), f"({C['baseline_ok'].mean():.2%})")
print("Segments with hourly rows:", C["segment_id"].nunique(), "(expected 4488)")
print("\nRows per year:")
print(C.groupby(C["hour_start"].dt.year).size().to_string())
print("\nVolume by source:")
print(C.groupby("source")["volume"].describe(percentiles=[.5, .95, .99]).round(1).to_string())
assert (C["source"] == "automated").sum() == len(A)
assert (C["source"] == "historical").sum() == len(Hk)
assert abs(C.loc[C["source"] == "automated", "volume"].sum() - A["volume"].sum()) < 1e-3
assert abs(C.loc[C["source"] == "historical", "volume"].sum() - Hk["volume"].sum()) < 1e-3
print("per-source reconciliation: OK")

# 6. Save
C = C.sort_values(["segment_id", "direction", "road_type", "hour_start"]).reset_index(drop=True)
C.to_parquet(f"{OUT}/combined_hourly_v1.parquet", index=False)
print("saved combined_hourly_v1.parquet", C.shape)

inputs: automated 444140 | historical 1007516 | expected combined 1429638
borough sources: {'automated': 2840, 'lion': 1578, 'none': 67, 'manual_override': 3}
boro_disagreement segments: 16



Historical rows overlapping automated: 22018 {'unspecified': 22018}
combined rows: 1429638 | expected: 1429638

Duplicate keys at (segment, direction, road_type, hour): 0 (expected 0)
Source counts: {'historical': 985498, 'automated': 444140}
Road types: {'unspecified': 1424550, 'service': 3024, 'main': 2064}
Identity-conflict rows: 12148 (0.85%)
Baseline-eligible rows: 1412402 (98.79%)
Segments with hourly rows: 4488 (expected 4488)

Rows per year:
hour_start
2000       472
2006       664
2007      3885
2008      8988
2009     27251
2010     29813
2011     28241
2012    210188
2013     58595
2014    172961
2015    166720
2016    105794
2017    149040
2018     47131
2019    162623
2020    141243
2021     32970
2022     18744
2023     18419
2024     23405
2025     20218
2026      2273

Volume by source:
               count   mean    std  min    50%     95%     99%      max
source                                                                 
automated   444140.0  466.6  664.5  0.0  

In [9]:

OUT = str(DATA_ROOT / "clean")
C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet",
                    columns=["segment_id", "direction", "road_type", "hour_start", "date",
                             "source", "baseline_ok"])
C["month"] = C["hour_start"].dt.month
C["year"] = C["hour_start"].dt.year
C["sd"] = C["segment_id"].astype(str) + "|" + C["direction"]
SEASON = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
C["season"] = C["month"].map(SEASON)
B = C[C["baseline_ok"]].copy()

# 1. Rows by month and source (all rows)
print("Rows by month and source:")
print(pd.crosstab(C["month"], C["source"], margins=True).to_string())

# 2. Coverage by month, baseline rows: distinct segments, segment-directions, and segment-direction-days
dd = B.drop_duplicates(["sd", "date"]).groupby("month").size().rename("seg_dir_days")
m = (B.groupby("month").agg(segments=("segment_id", "nunique"), seg_dirs=("sd", "nunique"))
       .join(dd))
print("\nBaseline coverage by month:")
print(m.to_string())

# 3. Year x month heatmap of segment-direction-days (baseline rows)
hm = (B.drop_duplicates(["sd", "date"])
        .assign(year=lambda x: x["date"].dt.year)
        .pivot_table(index="year", columns="month", values="sd", aggfunc="size", fill_value=0)
        .reindex(columns=range(1, 13), fill_value=0))
print("\nSegment-direction-days, year x month (baseline):")
print(hm.to_string())

# 4. Seasons observed per segment-direction
sd_season = (B.groupby(["sd", "season"]).size().unstack(fill_value=0)
               .reindex(columns=["DJF", "MAM", "JJA", "SON"], fill_value=0))
has = sd_season > 0
print("\nSegment-directions by number of seasons observed:")
print(has.sum(axis=1).value_counts().sort_index().to_string())
print("\nSeasons observed:")
print(has.sum().to_string())

# 5. Paired-season counts: the basis for any seasonal factor
print("\nSegment-directions observed in both seasons:")
for a, b in [("JJA", "SON"), ("JJA", "MAM"), ("DJF", "JJA"), ("MAM", "SON"), ("DJF", "SON")]:
    print(f"  {a} and {b}: {int((has[a] & has[b]).sum())}")

# 6. Which source provides summer, and how much of the baseline is each season
print("\nBaseline rows by season and source:")
print(B.groupby(["season", "source"]).size().unstack(fill_value=0).to_string())
print("\nBaseline share by season (%):")
print((B["season"].value_counts(normalize=True) * 100).round(1).to_string())

Rows by month and source:
source  automated  historical      All
month                                 
1           32849       40347    73196
2           30539       63704    94243
3           45437       18654    64091
4           43499       15895    59394
5           46536       11641    58177
6           43783           0    43783
7           27315           0    27315
8           30863           0    30863
9           36502      143171   179673
10          44911      473667   518578
11          39195      218303   257498
12          22711         116    22827
All        444140      985498  1429638

Baseline coverage by month:
       segments  seg_dirs  seg_dir_days
month                                  
1           540       786          3336
2           596       881          4247
3           413       510          2954
4           319       388          2729
5           357       428          2718
6           347       399          2140
7           273       301          1408


In [10]:
OUT = str(DATA_ROOT / "clean")
C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet",
                    columns=["segment_id", "direction", "hour_start", "baseline_ok"])
C = C[C["baseline_ok"]].copy()
C["year"] = C["hour_start"].dt.year
SEASON = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
C["season"] = C["hour_start"].dt.month.map(SEASON)
C["sd"] = C["segment_id"].astype(str) + "|" + C["direction"]

has = (C.groupby(["sd", "year", "season"]).size().unstack("season", fill_value=0)
         .reindex(columns=["DJF", "MAM", "JJA", "SON"], fill_value=0) > 0)

print("Same-year pairs (segment-directions observed in both seasons within one year):")
for a, b in [("JJA", "SON"), ("JJA", "MAM"), ("DJF", "SON"), ("MAM", "SON"), ("DJF", "JJA")]:
    p = has[has[a] & has[b]].index.get_level_values("sd").unique()
    print(f"  {a}-{b}: {len(p)} segment-directions")

print("\nSame-year JJA-SON pairs by year:")
jp = has[has["JJA"] & has["SON"]].reset_index()
print(jp.groupby("year")["sd"].nunique().to_string())

Same-year pairs (segment-directions observed in both seasons within one year):
  JJA-SON: 60 segment-directions
  JJA-MAM: 54 segment-directions
  DJF-SON: 54 segment-directions
  MAM-SON: 16 segment-directions
  DJF-JJA: 0 segment-directions

Same-year JJA-SON pairs by year:
year
2009    9
2010    4
2011    4
2012    5
2013    4
2014    2
2015    4
2016    4
2017    4
2019    6
2020    6
2021    8


In [11]:
OUT = str(DATA_ROOT / "clean")
C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet",
                    columns=["segment_id", "direction", "hour_start", "volume", "baseline_ok"])
C = C[C["baseline_ok"]].copy()
C["year"] = C["hour_start"].dt.year
C["hr"] = C["hour_start"].dt.hour
SEASON = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
C["season"] = C["hour_start"].dt.month.map(SEASON)
C["sd"] = C["segment_id"].astype(str) + "|" + C["direction"]

# Mean volume per road, year, season and hour of day
mh = C.groupby(["sd", "year", "season", "hr"])["volume"].mean().unstack("season")
mh = mh[["JJA", "SON"]].dropna()          # only hours observed in both seasons

# Ratio per road-year, using matched hours (require at least 12 matched hours)
r = mh.groupby(["sd", "year"]).agg(jja=("JJA", "sum"), son=("SON", "sum"), n_hours=("JJA", "size"))
r = r[(r["n_hours"] >= 12) & (r["son"] > 0)].copy()
r["ratio"] = r["jja"] / r["son"]
r = r.reset_index()
print("Road-years with a usable ratio:", len(r), "| distinct roads:", r["sd"].nunique())
print(r["ratio"].describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())

# Median across roads, with a bootstrap interval
rng = np.random.default_rng(0)
boots = [np.median(rng.choice(r["ratio"].to_numpy(), len(r))) for _ in range(2000)]
print(f"\nMedian summer/autumn ratio: {r['ratio'].median():.3f}  "
      f"95% bootstrap CI: [{np.percentile(boots, 2.5):.3f}, {np.percentile(boots, 97.5):.3f}]")

# By year
print("\nBy year:")
print(r.groupby("year")["ratio"].agg(["median", "size"]).round(3).to_string())

# By borough
seg = pd.read_parquet(f"{OUT}/segment_table_v2.parquet", columns=["segment_id", "borough"])
r["segment_id"] = r["sd"].str.split("|").str[0].astype(int)
r = r.merge(seg, on="segment_id", how="left")
print("\nBy borough:")
print(r.groupby("borough")["ratio"].agg(["median", "size"]).round(3).to_string())

Road-years with a usable ratio: 50 | distinct roads: 50
count    50.000
mean      1.077
std       0.370
min       0.562
10%       0.746
25%       0.904
50%       1.041
75%       1.106
90%       1.211
max       2.657

Median summer/autumn ratio: 1.041  95% bootstrap CI: [0.975, 1.079]

By year:
      median  size
year              
2009   0.942     5
2010   1.101     4
2011   0.756     4
2012   1.130     5
2013   0.706     1
2014   1.068     2
2015   1.058     4
2016   1.050     4
2017   0.876     1
2019   1.332     6
2020   0.944     6
2021   1.080     8

By borough:
               median  size
borough                    
Brooklyn        1.054    20
Manhattan       0.964     4
Queens          0.984    20
Staten Island   1.087     6


In [12]:
OUT = str(DATA_ROOT / "clean")
C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet",
                    columns=["segment_id", "direction", "hour_start", "volume", "baseline_ok", "source"])
C = C[C["baseline_ok"]].copy()
C["year"] = C["hour_start"].dt.year
C["hr"] = C["hour_start"].dt.hour
SEASON = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
C["season"] = C["hour_start"].dt.month.map(SEASON)
C["sd"] = C["segment_id"].astype(str) + "|" + C["direction"]

# Rebuild the per-road-year ratios, keeping the source of each season
mh = C.groupby(["sd", "year", "season", "hr"])["volume"].mean().unstack("season")
mh = mh[["JJA", "SON"]].dropna()
r = mh.groupby(["sd", "year"]).agg(jja=("JJA", "sum"), son=("SON", "sum"), n_hours=("JJA", "size"))
r = r[(r["n_hours"] >= 12) & (r["son"] > 0)].copy()
r["ratio"] = r["jja"] / r["son"]
r = r.reset_index()

src = (C.groupby(["sd", "year", "season"])["source"]
         .agg(lambda s: "+".join(sorted(s.unique()))).unstack("season"))
src = src[["JJA", "SON"]].rename(columns={"JJA": "jja_src", "SON": "son_src"}).reset_index()
r = r.merge(src, on=["sd", "year"], how="left")

print("Source combinations (summer, autumn) across usable road-years:")
print(pd.crosstab(r["jja_src"], r["son_src"]).to_string())

print("\nLargest ratios (check these for campaign artifacts):")
print(r.nlargest(5, "ratio")[["sd", "year", "jja", "son", "n_hours", "ratio", "jja_src", "son_src"]].to_string())

# Year-clustered bootstrap: resample years first, then all roads within each sampled year
years = r["year"].unique()
rng = np.random.default_rng(1)
boots = []
for _ in range(4000):
    ys = rng.choice(years, size=len(years), replace=True)
    boots.append(pd.concat([r[r["year"] == y] for y in ys])["ratio"].median())
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"\nYear-clustered 95% CI for the median ratio: [{lo:.3f}, {hi:.3f}]")
print("Median:", round(r["ratio"].median(), 3), "| road-years:", len(r), "| years:", len(years))

# Ratio with the largest outlier removed, for sensitivity
trim = r[r["ratio"] < r["ratio"].quantile(0.98)]
print("Median without top 2% of ratios:", round(trim["ratio"].median(), 3))

Source combinations (summer, autumn) across usable road-years:
son_src    automated
jja_src             
automated         50

Largest ratios (check these for campaign artifacts):
           sd  year           jja     son  n_hours     ratio    jja_src    son_src
3   113012|SB  2019  11194.250000  4213.0       24  2.657073  automated  automated
22  172473|WB  2019  16118.142857  6410.0       24  2.514531  automated  automated
20  171271|NB  2019   4992.875000  2927.5       24  1.705508  automated  automated
48   90518|WB  2012    406.900000   245.0       17  1.660816  automated  automated
27  188637|SB  2021   2780.000000  2246.0       16  1.237756  automated  automated

Year-clustered 95% CI for the median ratio: [0.942, 1.073]
Median: 1.041 | road-years: 50 | years: 12
Median without top 2% of ratios: 1.031


In [13]:
import pandas as pd, numpy as np

OUT = str(DATA_ROOT / "clean")
C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet",
                    columns=["segment_id", "direction", "road_type", "hour_start", "date",
                             "volume", "source", "study_id", "baseline_ok"])
C = C[C["baseline_ok"]].copy()
SEASON = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
C["year"] = C["hour_start"].dt.year
C["season"] = C["hour_start"].dt.month.map(SEASON)

def inspect(sid, dr):
    x = C[(C["segment_id"] == sid) & (C["direction"] == dr)]
    days = (x.groupby(["date", "year", "season", "study_id", "source"])
              .agg(total=("volume", "sum"), hours=("volume", "size")).reset_index())
    full = days[days["hours"] == 24].copy()

    print(f"\n=== segment {sid} {dr} ===")
    print("Full-day totals by year and season (median, days):")
    print(full.groupby(["year", "season"])["total"].agg(["median", "size"]).round(0).to_string())

    print("\n2019 by study and season:")
    s19 = (full[full["year"] == 2019].groupby(["study_id", "season"])
           .agg(first=("date", "min"), last=("date", "max"), days=("date", "nunique"),
                median_total=("total", "median"), min_total=("total", "min"), max_total=("total", "max"))
           .round(0))
    print(s19.to_string())

    med = full.groupby("study_id")["total"].transform("median")
    odd = full[(full["year"] == 2019) & ((full["total"] > 2 * med) | (full["total"] < 0.5 * med))]
    print("\n2019 full days far from their study's median (>2x or <0.5x):")
    print(odd[["date", "season", "study_id", "total"]].to_string(index=False)
          if len(odd) else "  none")

for sid, dr in [(113012, "SB"), (172473, "WB"), (171271, "NB")]:
    inspect(sid, dr)


=== segment 113012 SB ===
Full-day totals by year and season (median, days):
              median  size
year season               
2019 JJA     12438.0     8
     SON      4213.0     1

2019 by study and season:
                     first       last  days  median_total  min_total  max_total
study_id season                                                                
31031    JJA    2019-08-21 2019-08-29     8       12438.0     5561.0    12914.0
         SON    2019-09-02 2019-09-02     1        4213.0     4213.0     4213.0

2019 full days far from their study's median (>2x or <0.5x):
      date season  study_id  total
2019-08-25    JJA     31031 5561.0
2019-09-02    SON     31031 4213.0

=== segment 172473 WB ===
Full-day totals by year and season (median, days):
              median  size
year season               
2019 JJA     17616.0     7
     SON      6410.0     1

2019 by study and season:
                     first       last  days  median_total  min_total  max_total
study_i

In [14]:
OUT = str(DATA_ROOT / "clean")
C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet",
                    columns=["segment_id", "direction", "hour_start", "date", "volume", "baseline_ok"])
C = C[C["baseline_ok"]].copy()
C["year"] = C["hour_start"].dt.year
C["hr"] = C["hour_start"].dt.hour
SEASON = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
C["season"] = C["hour_start"].dt.month.map(SEASON)
C["sd"] = C["segment_id"].astype(str) + "|" + C["direction"]

# Full days (24 hours) per road, year and season, for JJA and SON only
days = C.groupby(["sd", "year", "season", "date"]).size().rename("h").reset_index()
fd = (days[days["h"] == 24].groupby(["sd", "year", "season"]).size()
        .unstack("season", fill_value=0)
        .reindex(columns=["JJA", "SON"], fill_value=0)
        .rename(columns={"JJA": "jja_days", "SON": "son_days"}))

# Ratio per road-year on matched hours (same method as before)
mh = C[C["season"].isin(["JJA", "SON"])].groupby(["sd", "year", "season", "hr"])["volume"].mean().unstack("season")
mh = mh[["JJA", "SON"]].dropna()
r = mh.groupby(["sd", "year"]).agg(jja=("JJA", "sum"), son=("SON", "sum"), n_hours=("JJA", "size"))
r = r[(r["n_hours"] >= 12) & (r["son"] > 0)].copy()
r["ratio"] = r["jja"] / r["son"]
r = r.join(fd, how="left").fillna({"jja_days": 0, "son_days": 0})

def summarise(df, label):
    rng = np.random.default_rng(1)
    years = df["year"].unique()
    boots = []
    for _ in range(4000):
        ys = rng.choice(years, size=len(years), replace=True)
        boots.append(pd.concat([df[df["year"] == y] for y in ys])["ratio"].median())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"{label}: road-years={len(df)} roads={df['sd'].nunique()} years={len(years)} "
          f"median={df['ratio'].median():.3f} year-clustered CI=[{lo:.3f}, {hi:.3f}]")

# 1. Reproduce the earlier result (no day filter)
summarise(r.reset_index(), "Before filter (should match 50 / 1.041)")

# 2. Apply the filter: at least 3 full days in each season, same road-year
f = r[(r["jja_days"] >= 3) & (r["son_days"] >= 3)].reset_index()
removed = r[~r.index.isin(f.set_index(["sd", "year"]).index)].reset_index()
print()
summarise(f, "After filter (>=3 full days per season)")
print(f"Removed road-years: {len(removed)}")
print(removed[["sd", "year", "jja_days", "son_days", "ratio"]].sort_values("ratio", ascending=False).to_string(index=False))

print("\nAfter filter, by year:")
print(f.groupby("year")["ratio"].agg(["median", "size"]).round(3).to_string())
print("\nAfter filter, trimmed median (top 2% removed):",
      round(f[f["ratio"] < f["ratio"].quantile(0.98)]["ratio"].median(), 3))
print("Autumn full days per usable road-year:",
      f["son_days"].value_counts().sort_index().to_dict())

Before filter (should match 50 / 1.041): road-years=50 roads=50 years=12 median=1.041 year-clustered CI=[0.942, 1.073]

After filter (>=3 full days per season): road-years=17 roads=17 years=6 median=1.031 year-clustered CI=[0.898, 1.128]
Removed road-years: 33
       sd  year  jja_days  son_days    ratio
113012|SB  2019       8.0       1.0 2.657073
172473|WB  2019       7.0       1.0 2.514531
171271|NB  2019       8.0       1.0 1.705508
 90518|WB  2012       1.0       0.0 1.660816
188637|SB  2021       3.0       0.0 1.237756
 38235|NB  2009       2.0       2.0 1.207598
142677|NB  2021       3.0       0.0 1.122968
  2271|WB  2012       5.0       0.0 1.122619
188637|NB  2021       3.0       2.0 1.106304
186282|SB  2021       3.0       0.0 1.102474
167637|WB  2014       1.0       2.0 1.086902
149068|NB  2015       2.0      28.0 1.084640
177559|WB  2015       1.0       7.0 1.064806
142544|EB  2021       3.0       0.0 1.057343
 71360|NB  2012       2.0       6.0 1.053014
239110|SB  2015    

In [15]:
OUT = str(DATA_ROOT / "clean")
C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet",
                    columns=["segment_id", "direction", "hour_start", "date", "volume", "baseline_ok"])
C = C[C["baseline_ok"]].copy()
C["year"] = C["hour_start"].dt.year
C["hr"] = C["hour_start"].dt.hour
SEASON = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
C["season"] = C["hour_start"].dt.month.map(SEASON)
C["sd"] = C["segment_id"].astype(str) + "|" + C["direction"]

# Full days (24 hours) and their counts per road-year and season
dd = C.groupby(["sd", "year", "season", "date"]).size().rename("h").reset_index()
full = dd[dd["h"] == 24][["sd", "year", "season", "date"]]
F = C.merge(full, on=["sd", "year", "season", "date"])
cnt = (full[full["season"].isin(["JJA", "SON"])]
       .groupby(["sd", "year", "season"]).size().unstack("season", fill_value=0)
       .reindex(columns=["JJA", "SON"], fill_value=0)
       .rename(columns={"JJA": "jja_days", "SON": "son_days"}).reset_index())

def ratios(rows, min_hours):
    mh = (rows[rows["season"].isin(["JJA", "SON"])]
          .groupby(["sd", "year", "season", "hr"])["volume"].mean().unstack("season"))
    mh = mh[["JJA", "SON"]].dropna()
    r = mh.groupby(["sd", "year"]).agg(jja=("JJA", "sum"), son=("SON", "sum"), n_hours=("JJA", "size"))
    r = r[(r["n_hours"] >= min_hours) & (r["son"] > 0)].copy()
    r["ratio"] = r["jja"] / r["son"]
    return r.reset_index()

def attach(r):
    return r.merge(cnt, on=["sd", "year"], how="left").fillna({"jja_days": 0, "son_days": 0})

def summ(df, reps=2000):
    years = df["year"].unique()
    rng = np.random.default_rng(1)
    boots = []
    for _ in range(reps):
        ys = rng.choice(years, size=len(years), replace=True)
        boots.append(pd.concat([df[df["year"] == y] for y in ys])["ratio"].median())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    trimmed = df[df["ratio"] < df["ratio"].quantile(0.98)]["ratio"].median()
    return dict(road_years=len(df), roads=df["sd"].nunique(), years=len(years),
                median=round(df["ratio"].median(), 3), ci_lo=round(lo, 3), ci_hi=round(hi, 3),
                trimmed=round(trimmed, 3))

# Test A and B together
results = []
for label, rows, minh in [("all days", C, 12), ("full days only", F, 24)]:
    r = attach(ratios(rows, minh))
    results.append({"ratio_method": label, "min_full_days": "none", **summ(r)})
    for t in [3, 5]:
        f = r[(r["jja_days"] >= t) & (r["son_days"] >= t)]
        results.append({"ratio_method": label, "min_full_days": t, **summ(f)})
print(pd.DataFrame(results).to_string(index=False))

# By year, full-days method, for each threshold
r_full = attach(ratios(F, 24))
for t in [3, 5]:
    f = r_full[(r_full["jja_days"] >= t) & (r_full["son_days"] >= t)]
    print(f"\nFull days only, threshold {t}: by year")
    print(f.groupby("year")["ratio"].agg(["median", "size"]).round(3).to_string())

  ratio_method min_full_days  road_years  roads  years  median  ci_lo  ci_hi  trimmed
      all days          none          50     50     12   1.041  0.942  1.073    1.031
      all days             3          17     17      6   1.031  0.898  1.128    1.024
      all days             5           5      5      3   1.015  0.735  1.130    1.003
full days only          none          38     38     11   1.002  0.924  1.036    0.994
full days only             3          17     17      6   1.015  0.898  1.098    1.015
full days only             5           5      5      3   1.015  0.735  1.123    1.003

Full days only, threshold 3: by year
      median  size
year              
2010   1.044     4
2012   1.133     2
2016   1.044     4
2017   0.876     1
2019   0.735     1
2020   0.990     5

Full days only, threshold 5: by year
      median  size
year              
2012   1.123     1
2019   0.735     1
2020   1.015     3
